# Teach Permit the tool contract on a free GPU

The recorded hosted probe completed **0/12 correct reports** and made **10 forbidden attempts**. This notebook prepares a small AgentTune supervised adapter from **authored training demonstrations**, then evaluates real model outputs through the gateway and AuditKIT. No training results are included in this notebook.

Choose a **free GPU runtime** in Colab. The default commands show plans. GPU execution requires the separate execution checkbox. The complete GPU dependency and memory path has not been run for Permit; allocation, installation or memory failures may stop the experiment.

Accept the access conditions yourself on [Tiny Aya Global](https://huggingface.co/CohereLabs/tiny-aya-global), then add a Hugging Face read token named `HF_TOKEN` to Colab Secrets and allow this notebook to read it. A Cohere API key does not grant access to these weights. Model and toolkit licenses apply independently. The notebook does not publish artifacts or submit an entry.


In [ ]:
from pathlib import Path
import os, sys, subprocess
from datetime import datetime, timezone

assert sys.version_info >= (3, 12), "Use a Python 3.12+ runtime."
PERMIT_CHECKOUT = Path("/content/permit-agent")
if not PERMIT_CHECKOUT.exists():
    subprocess.run(["git", "clone", "https://github.com/NSEVEjk0/permit-agent.git", str(PERMIT_CHECKOUT)], check=True)
os.chdir(PERMIT_CHECKOUT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-training.txt"], check=True)
PERMIT_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
PERMIT_RUNS = Path("runs") / ("colab-" + PERMIT_RUN_ID)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Run directory:", PERMIT_RUNS)


## Prepare the pinned toolkit sources and verify the teaching data

The teaching rows are rule-authored examples from the training split, not outputs from Tiny Aya. Their prompts receive invoice data and new authority only after the corresponding gateway result. AgentTune SFT uses completion-only loss on the next tool action or final statement. These CPU checks do not run a model.


In [ ]:
for name, url, revision in [
    ("agenttune", "https://github.com/Lexsi-Labs/AgentTune.git", "d4db4c12d52342227c4113a70183d0f64aa2f345"),
    ("auditkit", "https://github.com/Lexsi-Labs/AuditKIT.git", "71316da345784ff220d5ed59337441a59ab31526"),
]:
    destination = Path(".local") / (name + "-source")
    destination.parent.mkdir(exist_ok=True)
    if not destination.exists():
        subprocess.run(["git", "clone", url, str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", revision], check=True)
    os.environ[name.upper() + "_SOURCE_DIR"] = str(destination.resolve())
verify_env = dict(os.environ, PYTHONPATH=str((Path(".local/auditkit-source/src")).resolve()))
subprocess.run([sys.executable, "-m", "unittest", "tests.test_environment", "tests.test_hosted", "tests.test_toolkits", "tests.test_teaching", "-v"], env=verify_env, check=True)
subprocess.run([sys.executable, "scripts/teaching.py", "prepare", "--out", str(PERMIT_RUNS / "teaching")], check=True)


## Confirm weight access and inspect the bounded plan

Only check the license field after you have accepted the conditions yourself. Tokens stay in the runtime. The four-bit option lowers weight memory and uses float16 on a T4 or bfloat16 where supported. Both compared models use the same quantization, precision, prompts, backend and decoding.


In [ ]:
PERMIT_LICENSE_ACCEPTED = False #@param {type:"boolean"}
if PERMIT_LICENSE_ACCEPTED:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["PERMIT_MODEL_LICENSE_ACCEPTED"] = "true"

PERMIT_BASE = PERMIT_RUNS / "base-dev"
PERMIT_ADAPTER = PERMIT_RUNS / "teaching-adapter"
PERMIT_AFTER = PERMIT_RUNS / "adapter-dev"
PERMIT_COMMON = ["--load-in-4bit", "--wall-seconds", "1800"]
PERMIT_COMMANDS = [
    [sys.executable, "scripts/teaching.py", "evaluate", "--out", str(PERMIT_BASE), "--split", "dev", "--limit", "12", *PERMIT_COMMON],
    [sys.executable, "scripts/teaching.py", "train", "--out", str(PERMIT_ADAPTER), "--teaching", str(PERMIT_RUNS / "teaching"), "--max-steps", "60", *PERMIT_COMMON],
    [sys.executable, "scripts/teaching.py", "evaluate", "--out", str(PERMIT_AFTER), "--split", "dev", "--limit", "12", "--adapter", str(PERMIT_ADAPTER / "adapter"), *PERMIT_COMMON],
]
for command in PERMIT_COMMANDS:
    subprocess.run(command, check=True)


## Run the development study when the runtime is ready

This runs up to 60 SFT steps and one trial on the same 12 balanced development cases before and after. Each process has a 30-minute runtime cap; the notebook can take up to 90 minutes across the three runs. A process cap does not cap a paid provider's billing. Use the free runtime you selected.

Inspect the per-case failures and uncertainty. A development improvement is a tuning signal; the final claim needs the untouched held-out comparison below. No improvement is guaranteed. Failed or interrupted runs retain incomplete evidence.


In [ ]:
PERMIT_RUN_DEVELOPMENT_STUDY = False #@param {type:"boolean"}
if PERMIT_RUN_DEVELOPMENT_STUDY:
    assert PERMIT_LICENSE_ACCEPTED, "Accept model conditions and configure HF_TOKEN first."
    import torch
    assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
    print("GPU:", torch.cuda.get_device_name(0))
    for command in PERMIT_COMMANDS:
        subprocess.run([*command, "--execute"], check=True)
    subprocess.run([sys.executable, "scripts/teaching.py", "compare", str(PERMIT_BASE / "run.json"), str(PERMIT_AFTER / "run.json"), "--out", str(PERMIT_RUNS / "dev-comparison.json")], check=True)
else:
    print("Plan only. Select the execution checkbox to start the GPU study.")


## Final held-out comparison, after development decisions are frozen

Run this once you have settled the training configuration. It evaluates all 72 held-out cases on the matched local base and adapter. Do not train on these cases or use their results to choose another adapter. Each model run has a one-hour cap. The hosted API probe is not the baseline for this comparison.


In [ ]:
PERMIT_RUN_HELDOUT = False #@param {type:"boolean"}
if PERMIT_RUN_HELDOUT:
    assert PERMIT_RUN_DEVELOPMENT_STUDY, "Complete and inspect the development study first."
    for name, adapter in [("base-heldout", None), ("adapter-heldout", PERMIT_ADAPTER / "adapter")]:
        command = [sys.executable, "scripts/teaching.py", "evaluate", "--split", "heldout", "--limit", "72", "--out", str(PERMIT_RUNS / name), "--load-in-4bit", "--wall-seconds", "3600", "--execute"]
        if adapter: command += ["--adapter", str(adapter)]
        subprocess.run(command, check=True)
    subprocess.run([sys.executable, "scripts/teaching.py", "compare", str(PERMIT_RUNS / "base-heldout/run.json"), str(PERMIT_RUNS / "adapter-heldout/run.json"), "--out", str(PERMIT_RUNS / "heldout-comparison.json")], check=True)


## Keep the evidence for review

The run directory contains the adapter, teaching manifest, model trajectories, reports and comparisons. Download it from Colab's Files panel before the session ends. Review permitted artifacts, remove credentials and publish the allowed model/dataset collections on Hugging Face separately. The notebook performs no publication.

[Permit source](https://github.com/NSEVEjk0/permit-agent) · [AgentTune distillation API](https://github.com/Lexsi-Labs/AgentTune/blob/d4db4c12d52342227c4113a70183d0f64aa2f345/docs/user-guide/distillation.md) · [Lexsi rules](https://hackathon.lexsi.ai/)
